In [1]:
import pandas as pd


In [2]:
import duckdb
df_check = duckdb.sql("SELECT * FROM 'app_daily_metrics.csv' LIMIT 5").df()
df_check


,date,app_id,app_name,platform,region,reach_users,sessions_total,duration_total_seconds,session_mean_seconds,session_median_seconds
0,2025-06-02,A001,Vista 1,iOS,UK,2177,3375,852544.09,259.94,188.27
1,2025-06-02,A001,Vista 1,Android,UK,2673,4610,801294.04,179.12,140.50
2,2025-06-02,A002,Riviera 2,iOS,UK,12051,25104,5697754.97,225.56,178.59
3,2025-06-02,A002,Riviera 2,Android,UK,22462,22462,4186183.01,185.05,160.69
4,2025-06-02,A003,Quattro 3,iOS,UK,7292,14251,3913287.07,278.30,217.48


In [3]:
#To understand my data
df = pd.read_csv('app_daily_metrics.csv')
#df.shape()
df.head()
#df.tail()
#df.describe()
df.dtypes


date                       object
app_id                     object
app_name                   object
platform                   object
region                     object
reach_users                 int64
sessions_total              int64
duration_total_seconds    float64
session_mean_seconds      float64
session_median_seconds    float64
dtype: object

In [4]:
#Convert date to date data type

df['date'] = pd.to_datetime(df['date'])
df.dtypes

date                      datetime64[ns]
app_id                            object
app_name                          object
platform                          object
region                            object
reach_users                        int64
sessions_total                     int64
duration_total_seconds           float64
session_mean_seconds             float64
session_median_seconds           float64
dtype: object

In [5]:
df['date'].isnull().sum()

np.int64(0)

In [6]:
mind = df['date'].min()
print(mind)

2025-06-02 00:00:00


In [7]:
maxd = df['date'].max()
print(maxd)

2025-08-30 00:00:00


In [8]:
#Check count of unique dates
df['date'].nunique()

90

In [9]:
#To check for Completeness at the date level
#Check that the unique date values = the difference between the min and max date values
(maxd - mind).days + 1

90

In [10]:
#To check for completeness at the app/platform level

duckdb.sql("""
SELECT app_id
     --,app_name
    ,platform
    ,COUNT(DISTINCT date) AS Dates_Reported
FROM 'app_daily_metrics.csv'
GROUP BY app_id
     ,platform
ORDER BY Dates_Reported


""").df()



duckdb.sql("""
WITH date_spine AS (
    SELECT DISTINCT 
        date 
    FROM 'app_daily_metrics.csv'
),
app_platform AS (
    SELECT DISTINCT 
        app_id
        ,platform
    FROM 'app_daily_metrics.csv'
),
expected AS (
    SELECT 
        a.app_id
        ,a.platform
        ,d.date
    FROM app_platform a
    CROSS JOIN date_spine d
)
SELECT 
    e.app_id
    ,e.platform
    ,e.date
FROM expected e
LEFT JOIN 'app_daily_metrics.csv' actual
    ON e.app_id = actual.app_id
    AND e.platform = actual.platform
    AND e.date = actual.date
WHERE actual.app_id IS NULL
""").df()

,app_id,platform,date


In [11]:
#To check for duplicates across app_id, platform and date
duckdb.sql("""
SELECT app_id
    ,platform
    ,date
    ,COUNT(*) AS All_Count

FROM 'app_daily_metrics.csv'

GROUP BY app_id
    ,platform
    ,date

HAVING COUNT(*) > 1


""").df()

,app_id,platform,date,All_Count


In [12]:
#Completeness rule/alert for production - set alert for no missing row

completeness_alert = duckdb.sql("""
WITH date_spine AS (
    SELECT DISTINCT
        date
    FROM 'app_daily_metrics.csv'
),
app_platform AS (
    SELECT DISTINCT
        app_id
        ,platform
    FROM 'app_daily_metrics.csv'
),
expected AS (
    SELECT
        a.app_id
        ,a.platform
        ,d.date
    FROM app_platform a
    CROSS JOIN date_spine d
)
SELECT
    e.app_id
    ,e.platform
    ,e.date AS metric_date
    ,'completeness' AS alert_category
    ,'missing_row' AS alert_type
    ,'critical' AS severity
    ,NULL AS metric_name
    ,NULL AS observed_value
    ,NULL AS expected_value
    ,NULL AS deviation_score
    ,'Expected app_id + platform + date combination has no data' AS description
    ,CURRENT_TIMESTAMP AS detected_at
    
FROM expected e

LEFT JOIN 'app_daily_metrics.csv' actual
    ON e.app_id = actual.app_id
    AND e.platform = actual.platform
    AND e.date = actual.date
    
WHERE actual.app_id IS NULL
""").df()

print(completeness_alert.head())

Empty DataFrame
Columns: [app_id, platform, metric_date, alert_category, alert_type, severity, metric_name, observed_value, expected_value, deviation_score, description, detected_at]
Index: []


In [13]:
#Consistency check - does sessions_mean_seconds * sessions_total reconciles with duration_total-seconds?
#First, I looked at the distribution of the data to determine what the accepted percentage difference should be. Seeing that the max is 0.0309, I picked the percentage diff as 0.0400
#Percentage diff = ((sessions_mean_seconds * sessions_total) - duration_total_seconds) / duration_total_seconds
#Normally to determine this, it would be a decision made with stakeholders with more contectual knowledge.
#Sessions_total ranges from 1736 to 249722, which is a very wide range for a daily count
duckdb.sql("""
SELECT MIN(pct_diff) AS min_diff, MAX(pct_diff) AS max_diff, AVG(pct_diff) AS avg_diff
FROM (
    SELECT ABS(session_mean_seconds * sessions_total - duration_total_seconds) / duration_total_seconds AS pct_diff
    FROM 'app_daily_metrics.csv'
)
""").df()

,min_diff,max_diff,avg_diff
0,0.000003,0.030901,0.014814


In [14]:
#To create a flag column for avg_count
#I picked 0.0400 because it is close to double the average gap, comfortably above the max, without being so loose it'd miss a real 2x-sized anomaly
duckdb.sql("""
With Cal_dur AS
    (SELECT app_id
     ,platform
      ,date
      ,sessions_total
     ,duration_total_seconds
     ,session_mean_seconds
     ,session_median_seconds
     ,session_mean_seconds * sessions_total AS calculated_duration
    FROM 'app_daily_metrics.csv'
    )

SELECT *
     ,ABS(calculated_duration - duration_total_seconds) / duration_total_seconds AS diff_pct
     ,CASE
     WHEN ABS(calculated_duration - duration_total_seconds) / duration_total_seconds < 0.04 THEN 0
     WHEN ABS(calculated_duration - duration_total_seconds) / duration_total_seconds >= 0.04 THEN 1
     ELSE 2
     END AS avg_count_flag
FROM Cal_dur

--WHERE ABS(calculated_duration - duration_total_seconds) / duration_total_seconds >= 0.04 
ORDER BY avg_count_flag DESC;




--SELECT
--     COUNT(*)
 
--FROM 'app_daily_metrics.csv'

--WHERE ABS((session_mean_seconds * sessions_total) - duration_total_seconds) / duration_total_seconds >= 0.02 


""").df()

,app_id,platform,date,sessions_total,duration_total_seconds,session_mean_seconds,session_median_seconds,calculated_duration,diff_pct,avg_count_flag
0,A001,iOS,2025-06-02,3375,852544.09,259.94,188.27,877297.50,0.029035,0
1,A001,Android,2025-06-02,4610,801294.04,179.12,140.50,825743.20,0.030512,0
2,A002,iOS,2025-06-02,25104,5697754.97,225.56,178.59,5662458.24,0.006195,0
3,A002,Android,2025-06-02,22462,4186183.01,185.05,160.69,4156593.10,0.007068,0
4,A003,iOS,2025-06-02,14251,3913287.07,278.30,217.48,3966053.30,0.013484,0
...,...,...,...,...,...,...,...,...,...,...
3595,A018,Android,2025-08-30,29260,7525194.63,259.15,159.87,7582729.00,0.007646,0
3596,A019,iOS,2025-08-30,97500,22549191.02,225.93,165.69,22028175.00,0.023106,0
3597,A019,Android,2025-08-30,105091,21085223.99,197.94,168.84,20801712.54,0.013446,0
3598,A020,iOS,2025-08-30,22196,4302941.92,190.51,141.25,4228559.96,0.017286,0


In [15]:
#Consistency rule/alert for production - check that session_mean_seconds * sessions_total is close to the valure of duration_total_seconds
duration_mismatch_alert = duckdb.sql("""

With cal_dur AS (
    SELECT app_id, platform, date,
           duration_total_seconds, session_mean_seconds, session_median_seconds, sessions_total,
           session_mean_seconds * sessions_total AS calculated_duration
    FROM 'app_daily_metrics.csv'
),

diff AS (
    SELECT *,
           ABS(calculated_duration - duration_total_seconds) / duration_total_seconds AS pct_diff
    FROM cal_dur
)

SELECT
    app_id, platform, date AS metric_date,
    'consistency' AS alert_category,
    'duration_reconciliation_mismatch' AS alert_type,
    'warning' AS severity,
    'session_mean_seconds * sessions_total vs duration_total_seconds' AS metric_name,
    pct_diff AS observed_value,
    0.04 AS expected_value,
    NULL AS deviation_score,
    'Calculated duration from mean and session count does not reconcile with duration_total_seconds within 4% tolerance' AS description,
    CURRENT_TIMESTAMP AS detected_at
    
FROM diff

WHERE pct_diff >= 0.04


""").df()

print(duration_mismatch_alert.head())

Empty DataFrame
Columns: [app_id, platform, metric_date, alert_category, alert_type, severity, metric_name, observed_value, expected_value, deviation_score, description, detected_at]
Index: []


In [16]:
#Consistency check - does distinct reach_users equals or exceed sessions_total?
#
duckdb.sql("""
With tableCC AS
(SELECT app_id
    ,platform
    ,date
    ,reach_users
    ,sessions_total
    ,CASE
        WHEN reach_users <= sessions_total THEN 0
        ELSE 1
    END AS reach_users_flag

FROM 'app_daily_metrics.csv'
)

SELECT *

FROM tableCC

ORDER BY reach_users_flag DESC



""").df()





,app_id,platform,date,reach_users,sessions_total,reach_users_flag
0,A001,iOS,2025-06-02,2177,3375,0
1,A001,Android,2025-06-02,2673,4610,0
2,A002,iOS,2025-06-02,12051,25104,0
3,A002,Android,2025-06-02,22462,22462,0
4,A003,iOS,2025-06-02,7292,14251,0
...,...,...,...,...,...,...
3595,A018,Android,2025-08-30,13836,29260,0
3596,A019,iOS,2025-08-30,70366,97500,0
3597,A019,Android,2025-08-30,63664,105091,0
3598,A020,iOS,2025-08-30,9620,22196,0


In [17]:
#Consistency rule/alert for production - check if distinct reach_users equals or exceed sessions_total 
users_exceeds_sessions = duckdb.sql("""
WITH tableCC AS (
    SELECT app_id
            ,platform
            ,date
           ,reach_users
           ,sessions_total
           
    FROM 'app_daily_metrics.csv'
),

diff AS (
    SELECT *,
           reach_users > sessions_total AS users_exceeds_sessions
    FROM tableCC
)

SELECT  app_id
    ,platform 
    ,date AS metric_date,
    'consistency' AS alert_category,
    'users_exceeds_sessions' AS alert_type,
    'critical' AS severity,
    'reach_users vs sessions_total' AS metric_name,
    reach_users AS observed_value,
    sessions_total AS expected_value,
    NULL AS deviation_score,
    'reach_users exceeds sessions_total, which is not logically possible' AS description,
    CURRENT_TIMESTAMP AS detected_at
    
FROM diff

WHERE users_exceeds_sessions



""").df()

print(users_exceeds_sessions.head())


Empty DataFrame
Columns: [app_id, platform, metric_date, alert_category, alert_type, severity, metric_name, observed_value, expected_value, deviation_score, description, detected_at]
Index: []


In [18]:
#Consistency check - does session_median_seconds vs session_mean_seconds fall off a wide range
#

duckdb.sql("""
SELECT MIN(Avg_mean_medium)
    ,MAX(Avg_mean_medium)
    ,AVG(Avg_mean_medium)

FROM (SELECT
    session_mean_seconds/session_median_seconds AS Avg_mean_medium
    
    
FROM 'app_daily_metrics.csv'
)





""").df()

,min(Avg_mean_medium),max(Avg_mean_medium),avg(Avg_mean_medium)
0,0.64256,4.287311,1.359993


In [19]:
#To check the top 5 and bottom 5 rows for patterns out of the ordinary
duckdb.sql("""

SELECT *

FROM (SELECT app_id
    ,platform
    ,date
    ,sessions_total
    ,session_mean_seconds
    ,session_median_seconds,
    session_median_seconds / session_mean_seconds AS ratio

FROM 'app_daily_metrics.csv'

)

ORDER BY ratio DESC
--ORDER BY ratio 

LIMIT 5 

""").df()

,app_id,platform,date,sessions_total,session_mean_seconds,session_median_seconds,ratio
0,A017,Android,2025-08-16,70338,75.70,117.81,1.556275
1,A005,Android,2025-08-07,17126,69.70,106.03,1.521234
2,A011,Android,2025-07-31,25579,30.00,39.88,1.329333
3,A010,Android,2025-06-26,7561,128.59,163.13,1.268606
4,A020,iOS,2025-07-12,6242,110.38,137.75,1.247962


In [20]:
#since the observed ratio falls into 0.23 – 1.56 range, I set my flag threshhold as < 0.15 and > 1.65.
#
duckdb.sql("""
SELECT *
    ,CASE
        WHEN med_mean_ratio < 0.15 OR med_mean_ratio > 1.65 THEN 1
        ELSE 0
    END AS med_mean_flag

FROM (SELECT app_id
    ,platform
    ,date
    ,sessions_total
    ,session_mean_seconds
    ,session_median_seconds,
    session_median_seconds / session_mean_seconds AS med_mean_ratio

FROM 'app_daily_metrics.csv'

)

ORDER BY med_mean_flag DESC




""").df()

,app_id,platform,date,sessions_total,session_mean_seconds,session_median_seconds,med_mean_ratio,med_mean_flag
0,A001,iOS,2025-06-02,3375,259.94,188.27,0.724283,0
1,A001,Android,2025-06-02,4610,179.12,140.50,0.784390,0
2,A002,iOS,2025-06-02,25104,225.56,178.59,0.791763,0
3,A002,Android,2025-06-02,22462,185.05,160.69,0.868360,0
4,A003,iOS,2025-06-02,14251,278.30,217.48,0.781459,0
...,...,...,...,...,...,...,...,...
3595,A018,Android,2025-08-30,29260,259.15,159.87,0.616901,0
3596,A019,iOS,2025-08-30,97500,225.93,165.69,0.733369,0
3597,A019,Android,2025-08-30,105091,197.94,168.84,0.852986,0
3598,A020,iOS,2025-08-30,22196,190.51,141.25,0.741431,0


In [21]:
#Consistency rule/alert for production - does session_median_seconds vs session_mean_seconds fall off a wide range

med_mean_alert = duckdb.sql("""
With med_mean AS
(
    SELECT app_id, platform, date,
           sessions_total, session_mean_seconds, session_median_seconds,
           session_median_seconds / session_mean_seconds AS ratio
    FROM 'app_daily_metrics.csv'
)

SELECT
    app_id, platform, date AS metric_date,
    'consistency' AS alert_category,
    'median_mean_ratio_outlier' AS alert_type,
    'warning' AS severity,
    'session_median_seconds / session_mean_seconds' AS metric_name,
    ratio AS observed_value,
    NULL AS expected_value,
    NULL AS deviation_score,
    'Median_mean session length ratio outside expected 0.15-1.65 range' AS description,
    CURRENT_TIMESTAMP AS detected_at
    
FROM med_mean

WHERE ratio < 0.15 OR ratio > 1.65

""").df()


print(med_mean_alert.head())

Empty DataFrame
Columns: [app_id, platform, metric_date, alert_category, alert_type, severity, metric_name, observed_value, expected_value, deviation_score, description, detected_at]
Index: []


In [22]:
duckdb.sql("""
SELECT app_id, platform, date, reach_users, rolling_mean, rolling_stddev, z_score
FROM (
    SELECT *,
        (reach_users - rolling_mean) / rolling_stddev AS z_score
    FROM (
        SELECT app_id, platform, date, reach_users,
            AVG(reach_users) OVER (PARTITION BY app_id, platform ORDER BY date ROWS BETWEEN 28 PRECEDING AND 1 PRECEDING) AS rolling_mean,
            STDDEV(reach_users) OVER (PARTITION BY app_id, platform ORDER BY date ROWS BETWEEN 28 PRECEDING AND 1 PRECEDING) AS rolling_stddev
        FROM 'app_daily_metrics.csv'
    )
)
ORDER BY z_score DESC
LIMIT 10
""").df()

,app_id,platform,date,reach_users,rolling_mean,rolling_stddev,z_score
0,A004,Android,2025-06-04,55384,35915.000000,855.599205,22.754813
1,A005,Android,2025-08-03,21286,7747.107143,1111.533187,12.180377
2,A018,Android,2025-07-16,35005,12010.571429,2586.289727,8.890894
3,A004,Android,2025-06-19,118533,48874.882353,9022.145760,7.720793
4,A009,iOS,2025-06-14,76817,31254.083333,6082.508431,7.490810
5,A011,Android,2025-08-02,32283,16268.571429,2256.390914,7.097364
6,A006,Android,2025-07-06,80213,30566.214286,7457.901926,6.656937
7,A005,Android,2025-06-27,15610,7267.360000,1280.442563,6.515435
8,A016,Android,2025-07-31,116920,51270.821429,10543.263523,6.226647
9,A013,Android,2025-07-27,95731,36897.392857,9870.810141,5.960363


In [23]:
#Z_score giving untrustable numbers so I ensured that it require a minimum number of prior days before trusting a z-score

duckdb.sql("""

With windowed AS (
    SELECT app_id
        ,platform
        ,date
        ,reach_users,
        AVG(reach_users) OVER (PARTITION BY app_id, platform ORDER BY date ROWS BETWEEN 28 PRECEDING AND 1 PRECEDING) AS rolling_mean,
        STDDEV(reach_users) OVER (PARTITION BY app_id, platform ORDER BY date ROWS BETWEEN 28 PRECEDING AND 1 PRECEDING) AS rolling_stddev,
        COUNT(reach_users) OVER (PARTITION BY app_id, platform ORDER BY date ROWS BETWEEN 28 PRECEDING AND 1 PRECEDING) AS prior_days_count
    FROM 'app_daily_metrics.csv'
)
SELECT *,
    CASE
        WHEN prior_days_count >= 14 THEN (reach_users - rolling_mean) / rolling_stddev
        ELSE NULL
    END AS z_score
    
FROM windowed

ORDER BY app_id, platform, date

""").df()

,app_id,platform,date,reach_users,rolling_mean,rolling_stddev,prior_days_count,z_score
0,A001,Android,2025-06-02,2673,NaN,NaN,0,NaN
1,A001,Android,2025-06-03,2340,2673.000000,NaN,1,NaN
2,A001,Android,2025-06-04,2531,2506.500000,235.466558,2,NaN
3,A001,Android,2025-06-05,2501,2514.666667,167.099771,3,NaN
4,A001,Android,2025-06-06,2654,2511.250000,136.607406,4,NaN
...,...,...,...,...,...,...,...,...
3595,A020,iOS,2025-08-26,6682,7499.678571,1368.574713,28,-0.597467
3596,A020,iOS,2025-08-27,7534,7560.285714,1288.310000,28,-0.020403
3597,A020,iOS,2025-08-28,6216,7507.178571,1256.112258,28,-1.027917
3598,A020,iOS,2025-08-29,8727,7523.035714,1236.038220,28,0.974051


In [24]:
#To detect anomaly using rolling mean and std dev for the preceding 28 days.
#Added in the 14days - prior_days_count because it's the number of days that could exist and I can trust the Z_score

duckdb.sql("""
SELECT MIN(z_score) AS min_z
        ,MAX(z_score) AS max_z
        ,AVG(z_score) AS avg_z
        ,COUNT(*) AS valid_rows
        
FROM (
    WITH windowed AS (
        SELECT app_id, platform, date, reach_users,
            AVG(reach_users) OVER (PARTITION BY app_id, platform ORDER BY date ROWS BETWEEN 28 PRECEDING AND 1 PRECEDING) AS rolling_mean,
            STDDEV(reach_users) OVER (PARTITION BY app_id, platform ORDER BY date ROWS BETWEEN 28 PRECEDING AND 1 PRECEDING) AS rolling_stddev,
            COUNT(reach_users) OVER (PARTITION BY app_id, platform ORDER BY date ROWS BETWEEN 28 PRECEDING AND 1 PRECEDING) AS prior_days_count
        FROM 'app_daily_metrics.csv'
    )
    SELECT *,
        CASE
            WHEN prior_days_count >= 14 THEN (reach_users - rolling_mean) / rolling_stddev
            ELSE NULL
        END AS z_score
    FROM windowed
)
WHERE z_score IS NOT NULL
""").df()

,min_z,max_z,avg_z,valid_rows
0,-4.576757,12.180377,0.089053,3040


In [25]:
anomaly_alert = duckdb.sql("""
WITH windowed AS (
    SELECT app_id,
        platform,
        date,
        reach_users,
        AVG(reach_users) OVER (PARTITION BY app_id, platform ORDER BY date ROWS BETWEEN 28 PRECEDING AND 1 PRECEDING) AS rolling_mean,
        STDDEV(reach_users) OVER (PARTITION BY app_id, platform ORDER BY date ROWS BETWEEN 28 PRECEDING AND 1 PRECEDING) AS rolling_stddev,
        COUNT(reach_users) OVER (PARTITION BY app_id, platform ORDER BY date ROWS BETWEEN 28 PRECEDING AND 1 PRECEDING) AS prior_days_count
    FROM 'app_daily_metrics.csv'
),
scored AS (
    SELECT *,
        CASE
            WHEN prior_days_count >= 14 THEN (reach_users - rolling_mean) / rolling_stddev
            ELSE NULL
        END AS z_score
    FROM windowed
)
SELECT
    app_id, platform, date AS metric_date,
    'anomaly' AS alert_category,
    'reach_users_zscore_outlier' AS alert_type,
    CASE WHEN ABS(z_score) >= 5 THEN 'critical' ELSE 'warning' END AS severity,
    'reach_users rolling z-score' AS metric_name,
    reach_users AS observed_value,
    rolling_mean AS expected_value,
    z_score AS deviation_score,
    'reach_users is more than 3 standard deviations from its 28-day rolling baseline' AS description,
    CURRENT_TIMESTAMP AS detected_at
FROM scored
WHERE ABS(z_score) >= 3
""").df()

print(anomaly_alert.shape)


(51, 12)


In [26]:
#To confirm known genuine anomalies (traced back individually during development) appear in the final output

anomaly_alert[anomaly_alert['app_id'].isin(['A004', 'A005', 'A018'])]

,app_id,platform,metric_date,alert_category,alert_type,severity,metric_name,observed_value,expected_value,deviation_score,description,detected_at
13,A005,iOS,2025-08-12,anomaly,reach_users_zscore_outlier,warning,reach_users rolling z-score,1665,6311.357143,-3.970355,reach_users is more than 3 standard deviations...,2026-09-01 08:10:25.194507+01:00
15,A005,Android,2025-06-21,anomaly,reach_users_zscore_outlier,warning,reach_users rolling z-score,10636,6933.894737,3.588299,reach_users is more than 3 standard deviations...,2026-09-01 08:10:25.194507+01:00
16,A005,Android,2025-06-27,anomaly,reach_users_zscore_outlier,critical,reach_users rolling z-score,15610,7267.360000,6.515435,reach_users is more than 3 standard deviations...,2026-09-01 08:10:25.194507+01:00
17,A005,Android,2025-08-03,anomaly,reach_users_zscore_outlier,critical,reach_users rolling z-score,21286,7747.107143,12.180377,reach_users is more than 3 standard deviations...,2026-09-01 08:10:25.194507+01:00
27,A018,iOS,2025-07-13,anomaly,reach_users_zscore_outlier,warning,reach_users rolling z-score,16132,10489.571429,3.282611,reach_users is more than 3 standard deviations...,2026-09-01 08:10:25.194507+01:00
28,A018,iOS,2025-08-09,anomaly,reach_users_zscore_outlier,warning,reach_users rolling z-score,21147,10530.357143,3.742657,reach_users is more than 3 standard deviations...,2026-09-01 08:10:25.194507+01:00
33,A004,Android,2025-06-19,anomaly,reach_users_zscore_outlier,critical,reach_users rolling z-score,118533,48874.882353,7.720793,reach_users is more than 3 standard deviations...,2026-09-01 08:10:25.194507+01:00
38,A018,Android,2025-07-05,anomaly,reach_users_zscore_outlier,warning,reach_users rolling z-score,16994,11612.642857,3.127306,reach_users is more than 3 standard deviations...,2026-09-01 08:10:25.194507+01:00
39,A018,Android,2025-07-08,anomaly,reach_users_zscore_outlier,warning,reach_users rolling z-score,3595,11915.178571,-3.991580,reach_users is more than 3 standard deviations...,2026-09-01 08:10:25.194507+01:00
40,A018,Android,2025-07-16,anomaly,reach_users_zscore_outlier,critical,reach_users rolling z-score,35005,12010.571429,8.890894,reach_users is more than 3 standard deviations...,2026-09-01 08:10:25.194507+01:00


In [27]:
#To combine all alerts to one table so the DQ check for completeness, consistency and anomaly could be in a place

alerts_df = pd.concat([
        completeness_alert
        ,duration_mismatch_alert
        ,users_exceeds_sessions
        ,med_mean_alert
        ,anomaly_alert
], ignore_index= True)

alerts_df.insert(0, 'alert_id', range(1, len(alerts_df) + 1))

print(alerts_df.shape)
alerts_df




(51, 13)


,alert_id,app_id,platform,metric_date,alert_category,alert_type,severity,metric_name,observed_value,expected_value,deviation_score,description,detected_at
0,1,A002,Android,2025-08-19,anomaly,reach_users_zscore_outlier,warning,reach_users rolling z-score,6349.0,19793.107143,-3.484417,reach_users is more than 3 standard deviations...,2026-09-01 08:10:25.194507+01:00
1,2,A014,iOS,2025-06-19,anomaly,reach_users_zscore_outlier,warning,reach_users rolling z-score,8315.0,17569.705882,-3.043881,reach_users is more than 3 standard deviations...,2026-09-01 08:10:25.194507+01:00
2,3,A014,iOS,2025-08-04,anomaly,reach_users_zscore_outlier,warning,reach_users rolling z-score,5389.0,18514.892857,-3.100160,reach_users is more than 3 standard deviations...,2026-09-01 08:10:25.194507+01:00
3,4,A009,Android,2025-08-20,anomaly,reach_users_zscore_outlier,warning,reach_users rolling z-score,66700.0,39172.142857,4.980478,reach_users is more than 3 standard deviations...,2026-09-01 08:10:25.194507+01:00
4,5,A011,iOS,2025-07-10,anomaly,reach_users_zscore_outlier,warning,reach_users rolling z-score,17474.0,11843.357143,3.043646,reach_users is more than 3 standard deviations...,2026-09-01 08:10:25.194507+01:00
5,6,A011,iOS,2025-08-15,anomaly,reach_users_zscore_outlier,warning,reach_users rolling z-score,22071.0,12609.000000,4.553457,reach_users is more than 3 standard deviations...,2026-09-01 08:10:25.194507+01:00
6,7,A013,Android,2025-07-05,anomaly,reach_users_zscore_outlier,warning,reach_users rolling z-score,49123.0,32154.607143,3.317125,reach_users is more than 3 standard deviations...,2026-09-01 08:10:25.194507+01:00
7,8,A013,Android,2025-07-13,anomaly,reach_users_zscore_outlier,warning,reach_users rolling z-score,60664.0,33796.678571,4.623232,reach_users is more than 3 standard deviations...,2026-09-01 08:10:25.194507+01:00
8,9,A013,Android,2025-07-26,anomaly,reach_users_zscore_outlier,warning,reach_users rolling z-score,62378.0,35779.714286,3.105810,reach_users is more than 3 standard deviations...,2026-09-01 08:10:25.194507+01:00
9,10,A013,Android,2025-07-27,anomaly,reach_users_zscore_outlier,critical,reach_users rolling z-score,95731.0,36897.392857,5.960363,reach_users is more than 3 standard deviations...,2026-09-01 08:10:25.194507+01:00


In [28]:
#To check all columns in alerts_df are reflecting and complete with no extras

for name, df in [('completeness', completeness_alert), ('duration', duration_mismatch_alert),
                  ('reach_sessions', users_exceeds_sessions), ('med_mean', med_mean_alert),
                  ('anomaly', anomaly_alert)]:
    print(name, df.columns.tolist())

completeness ['app_id', 'platform', 'metric_date', 'alert_category', 'alert_type', 'severity', 'metric_name', 'observed_value', 'expected_value', 'deviation_score', 'description', 'detected_at']
duration ['app_id', 'platform', 'metric_date', 'alert_category', 'alert_type', 'severity', 'metric_name', 'observed_value', 'expected_value', 'deviation_score', 'description', 'detected_at']
reach_sessions ['app_id', 'platform', 'metric_date', 'alert_category', 'alert_type', 'severity', 'metric_name', 'observed_value', 'expected_value', 'deviation_score', 'description', 'detected_at']
med_mean ['app_id', 'platform', 'metric_date', 'alert_category', 'alert_type', 'severity', 'metric_name', 'observed_value', 'expected_value', 'deviation_score', 'description', 'detected_at']
anomaly ['app_id', 'platform', 'metric_date', 'alert_category', 'alert_type', 'severity', 'metric_name', 'observed_value', 'expected_value', 'deviation_score', 'description', 'detected_at']


In [29]:
#Check alerts_df
alerts_df.head()
#alerts_df['alert_category'].value_counts()
#alerts_df.dtypes

,alert_id,app_id,platform,metric_date,alert_category,alert_type,severity,metric_name,observed_value,expected_value,deviation_score,description,detected_at
0,1,A002,Android,2025-08-19,anomaly,reach_users_zscore_outlier,warning,reach_users rolling z-score,6349.0,19793.107143,-3.484417,reach_users is more than 3 standard deviations...,2026-09-01 08:10:25.194507+01:00
1,2,A014,iOS,2025-06-19,anomaly,reach_users_zscore_outlier,warning,reach_users rolling z-score,8315.0,17569.705882,-3.043881,reach_users is more than 3 standard deviations...,2026-09-01 08:10:25.194507+01:00
2,3,A014,iOS,2025-08-04,anomaly,reach_users_zscore_outlier,warning,reach_users rolling z-score,5389.0,18514.892857,-3.100160,reach_users is more than 3 standard deviations...,2026-09-01 08:10:25.194507+01:00
3,4,A009,Android,2025-08-20,anomaly,reach_users_zscore_outlier,warning,reach_users rolling z-score,66700.0,39172.142857,4.980478,reach_users is more than 3 standard deviations...,2026-09-01 08:10:25.194507+01:00
4,5,A011,iOS,2025-07-10,anomaly,reach_users_zscore_outlier,warning,reach_users rolling z-score,17474.0,11843.357143,3.043646,reach_users is more than 3 standard deviations...,2026-09-01 08:10:25.194507+01:00
